# Brain MRI Tumor Classifier

Junior computer-vision practice with the same step-by-step workflow as a basic scikit-learn notebook. **Learning only—not a clinical diagnostic tool.**


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import confusion_matrix, recall_score, accuracy_score, precision_score, f1_score
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from train import read_data, metrics
import joblib


## 1. Read images & inspect the dataset
Place labeled images in `data/yes` and `data/no` first.


In [ ]:
x, y, image_names = read_data(Path('data'))
df = pd.DataFrame({'image': image_names, 'label': y})
df.head()


In [ ]:
df.info()


In [ ]:
df['label'].value_counts().plot(kind='bar', title='MRI classes')
plt.xticks([0, 1], ['No Tumor', 'Tumor'], rotation=0)
plt.show()


## 2. Train-test split
A simple image-level split for learning. Real research must split by patient to prevent leakage.


In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.3, random_state=42, stratify=y
)


## 3. Pipeline & GridSearchCV
Here labels are numeric `0/1`, so `scoring='f1'` is valid.


In [ ]:
model = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('classifier', SVC(class_weight='balanced'))
])

p_g = {
    'classifier__C': [0.1, 1, 10],
    'classifier__kernel': ['linear', 'rbf']
}

grid = GridSearchCV(
    estimator=model, param_grid=p_g, scoring='f1', cv=3, n_jobs=-1, error_score='raise'
)
grid.fit(x_train, y_train)


In [ ]:
print(grid.best_params_)
print(grid.best_score_)


## 4. Predictions & metrics


In [ ]:
best_model = grid.best_estimator_
y_pred_train = best_model.predict(x_train)
y_pred_test = best_model.predict(x_test)
metrics(y_train, y_test, y_pred_train, y_pred_test)


## 5. Save the model


In [ ]:
Path('models').mkdir(exist_ok=True)
joblib.dump(best_model, 'models/brain_mri_model.joblib')
print('Saved model')
